In [ ]:
r'''
Kaggriculture | Adaptive Farm Intelligence — Shepherd's Ledger

Public source: https://www.kaggle.com/code/haideptry/the-shepherds-ledger-herd-safe-sovereign
Source snapshot: version 8/8. Kaggle currently shows 1929.8; its recorded best is 2341.0 on V1. Scores change over time and are not a performance guarantee.

Strategy: project shed stock after farm actions, detect cash-product SELLs that would execute zero units, and advance later executable SELLs into those queue gaps before prices decay. The published code and its integrity checks are retained.

The notebook's head-to-head figures are claims from the public author and have not been independently reproduced here. Its included simulation is self-play, not a comparison against current public opponents. Run top to bottom; nothing is submitted automatically.

Output: main.py and submission.tar.gz in /kaggle/working (or the current working directory locally). Verify the artifact and test against current opponents before submitting.
'''

In [ ]:
r'''
## 1 · Philosophy: Demand-Preserving Turn Sale Timing & Sovereign Hole-Closure

A forensic evaluation of competitive high-ladder match replays reveals the critical market execution breakthrough:
1. **The Ghost Order Problem**:
   Conventional agents place planned market SELL orders based on static forecasts. When preceding actions leave shed inventory insufficient, these orders execute zero units, leaving empty gaps in the market transaction queue while later viable sales are delayed.
2. **The Demand-Preserving Hole-Closure Solution**:
   - Computes exact projected shed inventory after physical unit actions are executed.
   - Identifies cash-product SELLs (`CARROT`, `TOMATO`, `STRAWBERRY`, `MELON`, `EGG`, `MILK`, `WOOL`) that would execute zero units and marks them as queue holes.
   - Pulls subsequent executable cash-product SELLs forward into earlier holes, capturing higher price tiers before market decay.
   - Non-SELL orders (`BUY_SEED`, `BUY_ANIMAL`, `BUY_LAND`, `HIRE`) and crop/livestock schedules remain completely untouched.

### The Demand-Preserving Architecture:
```
┌─────────────────────────────────────────────────────────────────────────────┐
│                 THE SHEPHERD'S LEDGER (DEMAND-PRESERVING SOVEREIGN)         │
└──────────────────────────────────────┬──────────────────────────────────────┘
                                       │
       ┌───────────────────────────────┼───────────────────────────────┐
       ▼                               ▼                               ▼
[PROJECTED SHED STOCK]       [QUEUE HOLE IDENTIFIER]     [EXECUTABLE SELL ADVANCE]
• Real-time post-farm stock  • Detects 0-execution SELLs • Pulls viable sales early
• FarmView unit simulation   • Zeros dead order slots    • Front-runs price glut
• Zero farm stall disruption • Isolates non-SELL orders  • Guarantees cash velocity
```
'''

In [ ]:
r'''
## 2 · The Sovereign Production Controller (`main.py`)

The next cell extracts and writes `main.py` (494,061 bytes, standard library only). It embodies the complete SOTA stack (`SHA-256: 36812d632b181fee7e0f249fa3a1eb364e0d373d6e4211f4e49803e8dd3582aa`).
'''

In [ ]:
import base64
import gzip
from pathlib import Path

# The Shepherd's Ledger: Demand-Preserving Sovereign Engine (Tetsutani 2026-09-26)
# Gzip + Base64 embedded (494,061 bytes uncompressed)
AGENT_B64 = """..."""

source = gzip.decompress(base64.b64decode(AGENT_B64))
Path("main.py").write_bytes(source)
print(f"✅ Generated main.py: {len(source):,} bytes successfully extracted.")

In [ ]:
r'''
## 2.1 · Automated Integrity & Dual Entrypoint Verification

Before evaluating, we verify file integrity, confirm standard-library-only imports, and ensure the production agent resolves strictly under Kaggle's last-callable rule.
'''

In [ ]:
import ast
import hashlib
import sys
from pathlib import Path

source = Path("main.py").read_bytes().replace(b"\r\n", b"\n")
Path("main.py").write_bytes(source)
digest = hashlib.sha256(source).hexdigest()

VALID_SHA256 = {
    "36812d632b181fee7e0f249fa3a1eb364e0d373d6e4211f4e49803e8dd3582aa"
}
assert digest in VALID_SHA256, f"Digest mismatch! Expected one of {VALID_SHA256}, got {digest}"

tree = ast.parse(source)
modules = sorted({a.name.split(".")[0] for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names}
                 | {n.module.split(".")[0] for n in ast.walk(tree) if isinstance(n, ast.ImportFrom) and n.module})
assert set(modules) <= set(sys.stdlib_module_names), f"Non-stdlib modules found: {set(modules) - set(sys.stdlib_module_names)}"

namespace = {}
exec(compile(source, "main.py", "exec"), namespace)
entry = [v for v in namespace.values() if callable(v)][-1]
assert entry.__name__ in ("agent", "kaggle_submission_agent", "ig_agent", "cha20_entry_agent", "kaggle_agent"), f"Entrypoint failed! Resolved to: {entry.__name__}"
del namespace

print(f"✅ Verified main.py: {len(source):,} bytes | {source.count(b'\n'):,} lines | SHA256: {digest[:16]}...")
print(f"✅ Stdlib modules: {', '.join(modules)}")
print(f"✅ Production entrypoint: {entry.__name__}() [OK]")

In [ ]:
r'''
## 3 · Head-to-Head Results Reported by the Public Author

The table below reproduces the source notebook's reported comparisons; this adaptation has not independently verified those results. The included 720-turn run is self-play and should not be interpreted as a current public-meta benchmark.
'''

In [ ]:
r'''
## 4 · Live 720-Turn Simulation & Visual Financial Audit

Simulating one complete 720-turn match in the official engine to inspect bank progression and compounding cash curves.
'''

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

try:
    from kaggle_environments import make
except ImportError:
    import subprocess, sys
    print("📦 Installing kaggle-environments==1.32.7...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle-environments==1.32.7"], check=True)
    from kaggle_environments import make

print("⚡ Simulating live 720-turn match...")
t0 = time.time()
env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 101})
env.run(["main.py", "main.py"])
elapsed = time.time() - t0
print(f"✅ Season completed in {elapsed:.2f}s!")

farms = [st[0]["observation"]["farms"] for st in env.steps]
days = np.arange(len(farms)) / 24.0
cash_0 = np.array([f[0]["money"] for f in farms]) / 1000.0
cash_1 = np.array([f[1]["money"] for f in farms]) / 1000.0
rewards = [st["reward"] for st in env.state]
print(f"🏆 Final Bank Balances: Seat 0: ${rewards[0]:,.0f} | Seat 1: ${rewards[1]:,.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.1, 1.2]})

# Subplot 1: Early-game surge
d_early = days[days <= 6.0]
c_early = cash_0[days <= 6.0] * 1000
ax1.plot(d_early, c_early, color="#31285c", lw=2.5, label="The Shepherd's Ledger Cash")
ax1.scatter([2.0], [c_early[int(2.0*24)]], color="#eab308", s=120, zorder=5)
ax1.annotate("Herd-Safe Pasture Lock\n(Guaranteed Feed Buffer)", xy=(2.0, c_early[int(2.0*24)]), xytext=(0.5, c_early[int(2.0*24)] + 400),
             arrowprops=dict(facecolor="#31285c", shrink=0.08, width=1.5, headwidth=7),
             fontsize=9.5, fontweight="bold", color="#31285c")
ax1.set_title("Days 0–6: Opening Capital Injection", fontweight="bold", fontsize=11.5)
ax1.set_xlabel("Season Day", fontsize=10.5)
ax1.set_ylabel("Bank Balance ($)", fontsize=10.5)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper left")

# Subplot 2: Full Season Progression
ax2.plot(days, cash_0, color="#120e24", lw=2.5, label="Seat 0 (Shepherd's Sovereign Farm)")
ax2.plot(days, cash_1, color="#9333ea", lw=1.8, linestyle="--", label="Seat 1 (Shepherd's Sovereign Farm)")
ax2.set_title("Full 30-Day Financial Compounding Trajectory", fontweight="bold", fontsize=11.5)
ax2.set_xlabel("Season Day (0 to 30)", fontsize=10.5)
ax2.set_ylabel("Bank Balance ($ thousands)", fontsize=10.5)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
r'''
## 5 · Packaging & One-Click Submission
    
The cell below builds the deterministic `submission.tar.gz` archive.

### How to Submit:
1. Click **Save Version &rarr; Save & Run All**.
2. Expand the **Output** tab in the right sidebar.
3. Click **Submit** next to `submission.tar.gz` to challenge the live Kaggle ladder!
'''

In [ ]:
import io
import tarfile
from pathlib import Path

source = Path("main.py").read_bytes()
buffer = io.BytesIO()

with tarfile.open(fileobj=buffer, mode="w:gz", format=tarfile.GNU_FORMAT) as tf:
    ti = tarfile.TarInfo("main.py")
    ti.size = len(source)
    ti.mtime = 1_700_000_000
    ti.mode = 0o644
    tf.addfile(ti, io.BytesIO(source))

archive_bytes = buffer.getvalue()
Path("submission.tar.gz").write_bytes(archive_bytes)

with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tf:
    members = tf.getmembers()
    assert len(members) == 1 and members[0].name == "main.py"
    extracted = tf.extractfile("main.py").read()
    assert extracted == source

print("📦 Packaged submission.tar.gz successfully!")
print(f"  • submission.tar.gz : {len(archive_bytes):,} bytes")
print(f"  • main.py           : {len(source):,} bytes")
print("\n🚀 Done! Submit submission.tar.gz directly from the Output pane.")